<a href="https://colab.research.google.com/github/Sangamithradillibabu/NLP/blob/main/CUSTOM_CHATBOT(NLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
from google.colab import files

uploaded = files.upload()


Saving Unit_4.pdf to Unit_4 (3).pdf


In [39]:
import PyPDF2

def read_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

pdf_text = read_pdf("/content/Unit_4.pdf")

print(pdf_text[:500])


1  
UNIT IV  MEMORY  AND  I/O ORGANIZATION  
Memory  Hierarchy, Memory  Chip Organization, Cache memory, Virtual memory. Parallel 
Bus Architectures, Internal Communication  Methodologies,  Serial  Bus Architectures, Mass 
storage, Input  and Output  Devices  
 
 
MEMORY  HIERARCHY  
A memory unit is considered as  a collection  of cells, in which cells is capable of  storing a  bit of 
information.  It stores  information  in group of  bits called byte or  word. Each  memory  location is 
ident


In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(pdf_text)

print("Chunks:", len(chunks))


Chunks: 76


In [41]:
!pip install -U langchain langchain-community faiss-cpu sentence-transformers


In [42]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_texts(chunks, embeddings)


In [44]:
print("PDF text length:", len(pdf_text))
print("Number of chunks:", len(chunks))
print("Vectorstore exists:", vectorstore is not None)


PDF text length: 54042
Number of chunks: 76
Vectorstore exists: True


In [50]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

docs = retriever.invoke("What is this PDF about?")
print("Retrieved docs:", len(docs))

for i, d in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(d.page_content[:300])


Retrieved docs: 4

--- Chunk 1 ---
distinguishes  read and write  transfers.  27  
 
Output  Interface  
 The operation  under control  of the handshake signals  valid and  idle in  a manner  similar to 
the handshake  used on the bus  with the Master -ready and  slave -ready signals.  
 When  it is ready  to accept  a character,  

--- Chunk 2 ---
Page Table  Base  register,  gives  the address  of the corresponding  entry  in the page  table.  (ie) it gives 
the starting  address  of the page  if that page  currently  resides in memory.  
Control  Bits in  Page Table:  The Control  bit specifies  the status  of the  page  while  it is in mai

--- Chunk 3 ---
. 
A simple  arrangement  for bus arbitration  using  a daisy  chain  
Sequence  of signals  during  transfer  of bus mastership  for the devices  
(ii) Distributed  Arbitration  
It means  that all devices  waiting  to use the bus have equal  responsibility  in carrying  out the 22  
arbitration  p

--- Chunk 4 ---
another pag

In [47]:
from transformers import pipeline

qa = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=256
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [56]:
def ask_question(question):
    docs = retriever.invoke(question)

    if not docs:
        return " Content not found in the PDF."

    context = "\n\n".join([d.page_content for d in docs])


    if len(context.strip()) < 50:
        return " Content not found in the PDF."

    prompt = f"""
You are a PDF-based assistant.

RULES:
- Answer ONLY using the given context.
- If the answer is not clearly present, say:
  "Content not found in the provided PDF."
- Give a clear, slightly detailed explanation (4–6 sentences).
- Do NOT add outside knowledge.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    result = qa(prompt)[0]["generated_text"]


    if "not found" in result.lower() or len(result.strip()) < 20:
        return " Content not found in the provided PDF."

    return f""" Answer:
{result}

 Relevant PDF Content:
{context[:800]}"""


In [ ]:
print("\nUNIT IV MEMORY AND I/O ORGANIZATION\n")
while True:
    q = input(" Ask a question (type exit to quit): ")
    if q.lower() == "exit":
        print("Thank you for using the PDF chatbot!")
        break

    print("\n Searching PDF...\n")
    response = ask_question(q)
    print(response)
    print("\n" + "="*70 + "\n")



UNIT IV MEMORY AND I/O ORGANIZATION


 Searching PDF...

 Answer:
Cache memory is a small -sized type of volatile computer memory that provides high speed data access to a processor and stores frequently used computer programs, applications and data.

 Relevant PDF Content:
data access  to a processor  and stores frequently used computer programs, applications  and data. 
Cache  memory  faster  than main memory. It  is also called CPU  memory.  This memory is  typically 
integrated directly  with the CPU chip or placed on a separate chip that has a separate bus 
interconnect  with the CPU  
 
 
When a  read request is received from the processor, the contents of  the memory location 
are transferred  into the cache  one word  at a time.  When  the program  references  any of the locations 
in this block,  the desired contents  are read directly  from the cache.  
The Cache  memory  stores  a reasonable number of  blocks  at a given  time but  this number 
is small compared to  the tot